# 14 - FAISS Hands-on

---

So far we have learned the major indexing techniques:

- Flat Index
- IVF
- PQ
- IVFPQ
- HNSW

Now we'll learn how these ideas are implemented using **FAISS**.

FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta for efficient similarity search over dense vectors.

## History

As AI systems grew,

they needed to search millions or billions of vectors efficiently.

Researchers at Meta developed FAISS

to provide fast and scalable vector search.

Instead of implementing algorithms from scratch,

developers can use FAISS to build production-ready similarity search systems.

In [1]:
import faiss
import numpy as np

In [2]:
np.random.seed(42)

dimension = 128

database = np.random.random(
    (10000,dimension)
).astype("float32")

query = np.random.random(
    (1,dimension)
).astype("float32")

## 1️⃣ IndexFlatL2

Theory

↓

Flat Index

Characteristics

- Exact Search
- No training
- Compare every vector

In [3]:
index = faiss.IndexFlatL2(dimension)

index.add(database)

distance, ids = index.search(query,5)

print(ids)

print(distance)

[[8769 9385   82 5125 9571]]
[[13.346977 14.548462 14.70829  14.756073 14.837166]]


### Observation

Advantages

- Exact results
- Easy to use

Limitations

- Slow for very large datasets

## 2️⃣ IndexIVFFlat

Theory

↓

IVF

Characteristics

- Clustering
- Faster search
- Approximate

In [4]:
nlist = 100

quantizer = faiss.IndexFlatL2(dimension)

index = faiss.IndexIVFFlat(
    quantizer,
    dimension,
    nlist
)

In [5]:
index.train(database)

index.add(database)

index.nprobe = 5

distance, ids = index.search(query,5)

print(ids)

[[8769 3948 4436 1056 7263]]


### Observation

Increasing nprobe

↓

Higher Recall

↓

Slower Search

## 3️⃣ IndexPQ

Theory

↓

Product Quantization

Characteristics

- Compressed vectors
- Memory efficient

In [6]:
m = 8

bits = 8

index = faiss.IndexPQ(
    dimension,
    m,
    bits
)

index.train(database)

index.add(database)

distance, ids = index.search(query,5)

print(ids)

[[8769 9571 4342 1983 7375]]


### Observation

PQ reduces memory,

but search becomes approximate.

## 4️⃣ IndexIVFPQ

Theory

↓

IVF

+

PQ

Characteristics

- Fast
- Memory efficient

In [7]:
quantizer = faiss.IndexFlatL2(dimension)

index = faiss.IndexIVFPQ(
    quantizer,
    dimension,
    100,
    8,
    8
)

index.train(database)

index.add(database)

index.nprobe = 5

distance, ids = index.search(query,5)

print(ids)

[[5148 9330 8769 9929 5774]]


### Observation

IVFPQ combines

- Faster search
- Lower memory

## 5️⃣ IndexHNSWFlat

Theory

↓

HNSW

Characteristics

- Graph-based search
- High Recall
- Very Fast

In [8]:
index = faiss.IndexHNSWFlat(
    dimension,
    16
)

index.add(database)

distance, ids = index.search(query,5)

print(ids)

[[5125 3654 9525 5389 2976]]


### Observation

HNSW often provides

excellent recall

with

very fast search.

##  Comparison

| FAISS Index | Theory | Exact | Memory | Speed |
|-------------|--------|-------|--------|-------|
| IndexFlatL2 | Flat | ✅ | High | Slow |
| IndexIVFFlat | IVF | ❌ | High | Fast |
| IndexPQ | PQ | ❌ | Low | Fast |
| IndexIVFPQ | IVF + PQ | ❌ | Low | Very Fast |
| IndexHNSWFlat | HNSW | ❌ | Medium | Very Fast |

## Choosing the Right Index

Small Dataset

↓

IndexFlatL2

Large Dataset

↓

IndexIVFFlat

Limited RAM

↓

IndexPQ

Large Dataset + Limited RAM

↓

IndexIVFPQ

High Recall

↓

IndexHNSWFlat

##  Think Like a Researcher

FAISS is a library.

It provides indexing algorithms.

But production applications need much more than an index.

They need:

- Metadata filtering
- CRUD operations
- Collections
- REST APIs
- Scalability
- Persistence

This leads us to **Vector Databases** such as Qdrant.